In [1]:
import os
import numpy as np
import pandas as pd
import librosa
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from collections import Counter

from sklearn.preprocessing import MultiLabelBinarizer, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import f1_score, classification_report

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [20]:
DEV_METADATA_PATH = "/content/drive/MyDrive/data/FSD50K.metadata/collection/collection_dev.csv"
EVAL_METADATA_PATH = "/content/drive/MyDrive/data/FSD50K.metadata/collection/collection_eval.csv"

dev = pd.read_csv(DEV_METADATA_PATH)
eval_df = pd.read_csv(EVAL_METADATA_PATH)

print("Train samples:", len(dev))
print("Eval samples:", len(eval_df))
dev.head()

Train samples: 40966
Eval samples: 10231


,fname,labels,mids
0,64760,Electric_guitar,/m/02sgy
1,16399,Electric_guitar,/m/02sgy
2,16401,Electric_guitar,/m/02sgy
3,16402,Electric_guitar,/m/02sgy
4,16404,Electric_guitar,/m/02sgy


In [21]:
AUDIO_DIR = "/content/drive/MyDrive/datasets/FSD50K/FSD50K.dev_audio"

print("Audio dir exists:", os.path.exists(AUDIO_DIR))
if os.path.exists(AUDIO_DIR):
    wav_files = [f for f in os.listdir(AUDIO_DIR) if f.endswith(".wav")]
    print("Number of wav files:", len(wav_files))
    print("First 5 files:", wav_files[:5])

Audio dir exists: True
Number of wav files: 40966
First 5 files: ['393010.wav', '23580.wav', '236571.wav', '408371.wav', '85124.wav']


In [22]:
#Create a initial sample
sample_df = dev.dropna(subset=["labels"]).sample(2000, random_state=42).copy()

print("Sample size:", len(sample_df))
sample_df.head()

Sample size: 2000


,fname,labels,mids
27012,332387,"Idling,Accelerating_and_revving_and_vroom","/m/07pb8fc,/m/07q2z82"
18406,81652,Snare_drum,/m/06rvn
22886,434250,"Fireworks,Firecracker","/m/0g6b5,/g/122z_qxw"
8288,117566,Ringtone,/m/01hnzm
40710,61605,Horse,/m/03k3r


In [23]:
sample_df["filepath"] = sample_df["fname"].astype(str).apply(
    lambda x: os.path.join(AUDIO_DIR, f"{x}.wav")
)

sample_df["exists"] = sample_df["filepath"].apply(os.path.exists)

print(sample_df["exists"].value_counts())
sample_df[["fname", "filepath", "exists"]].head()

exists
True    2000
Name: count, dtype: int64


,fname,filepath,exists
27012,332387,/content/drive/MyDrive/datasets/FSD50K/FSD50K....,True
18406,81652,/content/drive/MyDrive/datasets/FSD50K/FSD50K....,True
22886,434250,/content/drive/MyDrive/datasets/FSD50K/FSD50K....,True
8288,117566,/content/drive/MyDrive/datasets/FSD50K/FSD50K....,True
40710,61605,/content/drive/MyDrive/datasets/FSD50K/FSD50K....,True


In [24]:
# Empezamos unicamente con un ejemplo

valid_example = sample_df[sample_df["exists"] == True].iloc[0]
example_path = valid_example["filepath"]

print("Example path:", example_path)
print("Example labels:", valid_example["labels"])

Example path: /content/drive/MyDrive/datasets/FSD50K/FSD50K.dev_audio/332387.wav
Example labels: Idling,Accelerating_and_revving_and_vroom


In [25]:
#Creamos la funcion de features

def extract_handcrafted_features(filepath, sr=22050, n_mfcc=13):
    try:
        y, sr = librosa.load(filepath, sr=sr, mono=True)

        # MFCC
        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
        mfcc_mean = np.mean(mfcc, axis=1)
        mfcc_std = np.std(mfcc, axis=1)

        # Zero Crossing Rate
        zcr = librosa.feature.zero_crossing_rate(y)
        zcr_mean = np.mean(zcr, axis=1)
        zcr_std = np.std(zcr, axis=1)

        # Spectral Centroid
        spec_centroid = librosa.feature.spectral_centroid(y=y, sr=sr)
        spec_centroid_mean = np.mean(spec_centroid, axis=1)
        spec_centroid_std = np.std(spec_centroid, axis=1)

        # Chroma
        chroma = librosa.feature.chroma_stft(y=y, sr=sr)
        chroma_mean = np.mean(chroma, axis=1)
        chroma_std = np.std(chroma, axis=1)

        features = np.concatenate([
            mfcc_mean, mfcc_std,
            zcr_mean, zcr_std,
            spec_centroid_mean, spec_centroid_std,
            chroma_mean, chroma_std
        ])

        return features

    except Exception as e:
        return None

In [26]:
# Lo Probamos con un ejemplo
feat = extract_handcrafted_features(example_path)

print("Feature type:", type(feat))
print("Feature shape:", feat.shape)
print("First 10 values:", feat[:10])

Feature type: <class 'numpy.ndarray'>
Feature shape: (54,)
First 10 values: [-219.57571411  125.9070816    13.53838825   31.78510094    9.91389561
   32.85058975   12.43815231   15.32325172   -2.38303185    8.49230003]


In [27]:
# Ahora de toda la muestra

features = []
valid_rows = []

for _, row in tqdm(sample_df.iterrows(), total=len(sample_df)):
    if row["exists"]:
        feat = extract_handcrafted_features(row["filepath"])
        if feat is not None:
            features.append(feat)
            valid_rows.append(row)

X = np.array(features)
data_valid = pd.DataFrame(valid_rows).reset_index(drop=True)

print("X shape:", X.shape)
print("Valid rows:", len(data_valid))
data_valid.head()

  3%|▎         | 68/2000 [00:05<02:19, 13.84it/s]/usr/local/lib/python3.12/dist-packages/librosa/core/pitch.py:103: UserWarning: Trying to estimate tuning from empty frequency set.
  return pitch_tuning(
100%|██████████| 2000/2000 [11:04<00:00,  3.01it/s]

X shape: (2000, 54)
Valid rows: 2000


,fname,labels,mids,filepath,exists
0,332387,"Idling,Accelerating_and_revving_and_vroom","/m/07pb8fc,/m/07q2z82",/content/drive/MyDrive/datasets/FSD50K/FSD50K....,True
1,81652,Snare_drum,/m/06rvn,/content/drive/MyDrive/datasets/FSD50K/FSD50K....,True
2,434250,"Fireworks,Firecracker","/m/0g6b5,/g/122z_qxw",/content/drive/MyDrive/datasets/FSD50K/FSD50K....,True
3,117566,Ringtone,/m/01hnzm,/content/drive/MyDrive/datasets/FSD50K/FSD50K....,True
4,61605,Horse,/m/03k3r,/content/drive/MyDrive/datasets/FSD50K/FSD50K....,True


In [29]:
# Preparamos las etiqueteas
data_valid["label_list"] = data_valid["labels"].apply(
    lambda x: [label.strip() for label in str(x).split(",")]
)

print("First 5 label lists:")
display(data_valid[["labels", "label_list"]].head())

First 5 label lists:


,labels,label_list
0,"Idling,Accelerating_and_revving_and_vroom","[Idling, Accelerating_and_revving_and_vroom]"
1,Snare_drum,[Snare_drum]
2,"Fireworks,Firecracker","[Fireworks, Firecracker]"
3,Ringtone,[Ringtone]
4,Horse,[Horse]


In [30]:
# Nos quedamos con las mas frecuentes
all_labels = [label for sublist in data_valid["label_list"] for label in sublist]
label_counts = Counter(all_labels)

top_classes = [label for label, _ in label_counts.most_common(10)]

print("Top 10 classes:")
print(top_classes)

Top 10 classes:
['Laughter', 'Clarinet', 'Snare_drum', 'Cello', 'Piano', 'Bark', 'Flute', 'Acoustic_guitar', 'Violin_and_fiddle', 'Saxophone']


In [31]:
#Filtramos etiquetas a esas clases
all_labels = [label for sublist in data_valid["label_list"] for label in sublist]
label_counts = Counter(all_labels)

top_classes = [label for label, _ in label_counts.most_common(10)]

print("Top 10 classes:")
print(top_classes)

Top 10 classes:
['Laughter', 'Clarinet', 'Snare_drum', 'Cello', 'Piano', 'Bark', 'Flute', 'Acoustic_guitar', 'Violin_and_fiddle', 'Saxophone']


In [32]:
# Crear matriz multilabel Y
mlb = MultiLabelBinarizer(classes=top_classes)
Y = mlb.fit_transform(data_valid["label_list"])

print("Y shape:", Y.shape)
print("Number of classes:", len(mlb.classes_))
print("Classes:", mlb.classes_)

Y shape: (2000, 10)
Number of classes: 10
Classes: ['Laughter' 'Clarinet' 'Snare_drum' 'Cello' 'Piano' 'Bark' 'Flute'
 'Acoustic_guitar' 'Violin_and_fiddle' 'Saxophone']


/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_label.py:909: UserWarning: unknown class(es) ['Accelerating_and_revving_and_vroom', 'Accordion', 'Air_conditioning', 'Aircraft', 'Aircraft_engine', 'Alarm', 'Alarm_clock', 'Alto_saxophone', 'Animal', 'Applause', 'Artillery_fire', 'Babbling', 'Baby_cry_and_infant_cry', 'Baby_laughter', 'Bass_drum', 'Bass_guitar', 'Bassoon', 'Bathtub_(filling_or_washing)', 'Bee_and_wasp_and_etc.', 'Bell', 'Bicycle', 'Bicycle_bell', 'Bird', 'Bird_flight_and_flapping_wings', 'Bird_vocalization_and_bird_call_and_bird_song', 'Bleat', 'Blender', 'Boat_and_Water_vehicle', 'Boiling', 'Boom', 'Bowed_string_instrument', 'Brass_instrument', 'Breathing', 'Burping_and_eructation', 'Burst_and_pop', 'Bus', 'Buzz', 'Buzzer', 'Camera', 'Car', 'Car_alarm', 'Car_passing_by', 'Cash_register', 'Cat', 'Cattle_and_bovinae', 'Cellphone_buzz_and_vibrating_alert', 'Chant', 'Chatter', 'Cheering', 'Chewing_and_mastication', 'Chicken_and_rooster', 'Child_speech_and_kid_

In [33]:
# Separar train y set
X_train, X_test, y_train, y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (1600, 54)
X_test: (400, 54)
y_train: (1600, 10)
y_test: (400, 10)


In [34]:
# Escalar las features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [40]:
# Entrenar la MLP baseline
mlp = OneVsRestClassifier(
    MLPClassifier(
        hidden_layer_sizes=(128, 64),
        max_iter=100,
        random_state=42
    )
)

mlp.fit(X_train, y_train)

OneVsRestClassifier(estimator=MLPClassifier(hidden_layer_sizes=(128, 64),
                                            max_iter=100, random_state=42))

In [41]:
y_prob = mlp.predict_proba(X_test)

# probar con un umbral más bajo
threshold = 0.15
y_pred = (y_prob >= threshold).astype(int)


micro_f1 = f1_score(y_test, y_pred, average="micro", zero_division=0)
macro_f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)

print("Micro-F1:", micro_f1)
print("Macro-F1:", macro_f1)

Micro-F1: 0.45
Macro-F1: 0.44763791763791766


In [42]:
print("Number of predicted positives:", y_pred.sum())
print("Total prediction entries:", y_pred.size)
print("Percentage predicted positive:", y_pred.sum() / y_pred.size)

Number of predicted positives: 50
Total prediction entries: 4000
Percentage predicted positive: 0.0125
